In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.optimize import minimize

class ERA_PartialFractions(nn.Module):
    """基于部分分式分解的ERA激活函数"""
    def __init__(self, epsilon=1e-6):
        super(ERA_PartialFractions, self).__init__()
        self.epsilon = epsilon
        
        # 线性部分参数
        self.alpha = nn.Parameter(torch.randn(1))
        self.beta = nn.Parameter(torch.randn(1))
        
        # 部分分数参数
        self.p1 = nn.Parameter(torch.randn(1))
        self.q1 = nn.Parameter(torch.randn(1))
        self.p2 = nn.Parameter(torch.randn(1))
        self.q2 = nn.Parameter(torch.randn(1))
        
        # 分母参数 (确保初始时为正)
        self.c1 = nn.Parameter(torch.randn(1))
        self.d1 = nn.Parameter(torch.randn(1).abs() + 0.1)
        self.c2 = nn.Parameter(torch.randn(1))
        self.d2 = nn.Parameter(torch.randn(1).abs() + 0.1)
    
    def forward(self, x):
        # 分母多项式，确保不为零
        Q1 = (x - self.c1)**2 + self.d1**2 + self.epsilon
        Q2 = (x - self.c2)**2 + self.d2**2 + self.epsilon
        
        # 部分分数项
        frac1 = (self.p1 * x + self.q1) / Q1
        frac2 = (self.p2 * x + self.q2) / Q2
        
        # 完整ERA函数
        return self.alpha * x + self.beta + frac1 + frac2


class RationalNNWithERA(nn.Module):
    """结合ERA激活函数的有理神经网络"""
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        # 神经网络部分
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        
        # ERA激活函数
        self.era = ERA_PartialFractions()
        
        # 中间变量参数
        self.alpha = nn.Parameter(torch.randn(hidden_dim))
        self.beta = nn.Parameter(torch.randn(1))
        
        # 有理函数参数
        self.a = nn.Parameter(torch.randn(2))  # 线性分子
        self.b = nn.Parameter(torch.randn(2))  # 线性分母
        self.q1 = nn.Parameter(torch.randn(1))
        self.q2 = nn.Parameter(torch.ones(1))  # 初始为正，避免除零
        
        # 初始化策略：近似Swish函数
        self.initialize_as_swish()
    
    def initialize_as_swish(self):
        """将ERA初始化为近似Swish函数"""
        with torch.no_grad():
            # 近似Swish函数 x * sigmoid(x)
            self.era.alpha.fill_(0.5)   # 线性系数
            self.era.beta.fill_(0)      # 偏置
            
            # 部分分数参数设置
            self.era.p1.fill_(0.1)
            self.era.q1.fill_(0)
            self.era.p2.fill_(0)
            self.era.q2.fill_(0)
            
            # 分母参数设置为远离0的值，保证稳定性
            self.era.c1.fill_(0)
            self.era.d1.fill_(1)
            self.era.c2.fill_(0)
            self.era.d2.fill_(1)
            
            # 有理函数初始化为近似恒等映射
            self.a.data = torch.tensor([0.0, 1.0])
            self.b.data = torch.tensor([1.0, 0.0])
            self.q1.fill_(0.0)
            self.q2.fill_(1.0)
    
    def forward(self, x):
        # 前向传播：全连接 -> ERA激活 -> 中间变量 -> 有理函数输出
        h = self.era(self.fc1(x))
        z = torch.matmul(h, self.alpha) + self.beta  # 计算中间变量z
        
        # 有理函数部分
        p1 = self.a[0] + self.a[1] * z  # 线性分子
        p2 = self.b[0] + self.b[1] * z + 1e-6  # 线性分母，加小值避免除零
        
        return (p1 + self.q1) / (p2 + self.q2)
    
    def get_z(self, X):
        """获取隐藏层输出z，用于有理函数参数优化"""
        with torch.no_grad():
            h = self.era(self.fc1(X))
            z = torch.matmul(h, self.alpha) + self.beta
        return z
    
    def rational_params(self):
        """返回有理函数参数供SciPy优化"""
        return torch.cat([self.a.data, self.b.data, self.q1.data, self.q2.data]).numpy()
    
    def update_rational_params(self, new_params):
        """更新有理函数参数"""
        with torch.no_grad():
            self.a.data = torch.from_numpy(new_params[:2]).float()
            self.b.data = torch.from_numpy(new_params[2:4]).float()
            self.q1.data = torch.from_numpy(new_params[4:5]).float()
            self.q2.data = torch.from_numpy(new_params[5:6]).float()


def alt_train_with_era(model, X, y_true, max_iter=100, verbose=True):
    """交替训练模型：先优化神经网络参数，再优化有理函数参数"""
    # 确保输入是torch张量
    if not isinstance(X, torch.Tensor):
        X = torch.tensor(X, dtype=torch.float32)
    if not isinstance(y_true, torch.Tensor):
        y_true = torch.tensor(y_true, dtype=torch.float32)
    
    # 阶段一优化器 - 只优化神经网络参数(不包括有理函数参数)
    optimizer = optim.Adam([
        {'params': model.fc1.parameters()},
        {'params': model.alpha},
        {'params': model.beta},
        {'params': model.era.parameters()}  # 包含ERA参数
    ], lr=1e-3)
    
    # 记录损失
    losses = []
    
    for i in range(max_iter):
        # 阶段一：ADAM优化神经网络参数和ERA参数
        optimizer.zero_grad()
        y_pred = model(X)
        loss = torch.max(torch.abs(y_true - y_pred))  # 极小极大损失
        loss.backward()
        
        # 权重裁剪，稳定训练
        torch.nn.utils.clip_grad_value_(model.parameters(), clip_value=1.0)
        
        optimizer.step()
        
        # 阶段二：拟凸优化有理函数参数
        z = model.get_z(X).detach().numpy()
        
        # 定义优化目标函数：最小化最大误差
        def max_error(params):
            model.update_rational_params(params)
            with torch.no_grad():
                # 直接使用z计算，避免重复计算网络部分
                y_pred = (model.a[0] + model.a[1] * z + model.q1) / \
                         (model.b[0] + model.b[1] * z + model.q2 + 1e-6)
            return torch.max(torch.abs(torch.tensor(y_true.numpy() - y_pred.numpy()))).item()
        
        # 使用SLSQP优化器
        bounds = [(None, None)] * 6  # 所有参数无约束
        res = minimize(max_error, model.rational_params(), 
                      method='SLSQP', bounds=bounds, options={'maxiter': 50})
        
        # 更新有理函数参数
        model.update_rational_params(res.x)
        
        # 记录并打印损失
        losses.append(loss.item())
        if verbose and (i % 100 == 0):
            print(f"Iteration {i}, Loss: {loss.item():.6f}")
    
    return losses



In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from scipy.optimize import minimize
import matplotlib.pyplot as plt

class ERA_PartialFractions(nn.Module):
    """基于部分分式分解的ERA激活函数"""
    def __init__(self, epsilon=1e-6):
        super(ERA_PartialFractions, self).__init__()
        self.epsilon = epsilon
        
        # 线性部分参数
        self.alpha = nn.Parameter(torch.randn(1))
        self.beta = nn.Parameter(torch.randn(1))
        
        # 部分分数参数
        self.p1 = nn.Parameter(torch.randn(1))
        self.q1 = nn.Parameter(torch.randn(1))
        self.p2 = nn.Parameter(torch.randn(1))
        self.q2 = nn.Parameter(torch.randn(1))
        
        # 分母参数 (确保初始时为正)
        self.c1 = nn.Parameter(torch.randn(1))
        self.d1 = nn.Parameter(torch.randn(1).abs() + 0.1)
        self.c2 = nn.Parameter(torch.randn(1))
        self.d2 = nn.Parameter(torch.randn(1).abs() + 0.1)
    
    def forward(self, x):
        # 分母多项式，确保不为零
        Q1 = (x - self.c1)**2 + self.d1**2 + self.epsilon
        Q2 = (x - self.c2)**2 + self.d2**2 + self.epsilon
        
        # 部分分数项
        frac1 = (self.p1 * x + self.q1) / Q1
        frac2 = (self.p2 * x + self.q2) / Q2
        
        # 完整ERA函数
        return self.alpha * x + self.beta + frac1 + frac2


class RationalNNWithERA(nn.Module):
    """结合ERA激活函数的有理神经网络"""
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        # 神经网络部分
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        
        # ERA激活函数
        self.era = ERA_PartialFractions()
        
        # 中间变量参数
        self.alpha = nn.Parameter(torch.randn(hidden_dim))
        self.beta = nn.Parameter(torch.randn(1))
        
        # 有理函数参数
        self.a = nn.Parameter(torch.randn(2))  # 线性分子
        self.b = nn.Parameter(torch.randn(2))  # 线性分母
        self.q1 = nn.Parameter(torch.randn(1))
        self.q2 = nn.Parameter(torch.ones(1))  # 初始为正，避免除零
        
        # 初始化策略：近似Swish函数
        self.initialize_as_swish()
    
    def initialize_as_swish(self):
        """将ERA初始化为近似Swish函数"""
        with torch.no_grad():
            # 近似Swish函数 x * sigmoid(x)
            self.era.alpha.fill_(0.5)   # 线性系数
            self.era.beta.fill_(0)      # 偏置
            
            # 部分分数参数设置
            self.era.p1.fill_(0.1)
            self.era.q1.fill_(0)
            self.era.p2.fill_(0)
            self.era.q2.fill_(0)
            
            # 分母参数设置为远离0的值，保证稳定性
            self.era.c1.fill_(0)
            self.era.d1.fill_(1)
            self.era.c2.fill_(0)
            self.era.d2.fill_(1)
            
            # 有理函数初始化为近似恒等映射
            self.a.data = torch.tensor([0.0, 1.0])
            self.b.data = torch.tensor([1.0, 0.0])
            self.q1.fill_(0.0)
            self.q2.fill_(1.0)
    
    def forward(self, x):
        # 前向传播：全连接 -> ERA激活 -> 中间变量 -> 有理函数输出
        h = self.era(self.fc1(x))
        z = torch.matmul(h, self.alpha) + self.beta  # 计算中间变量z
        
        # 有理函数部分
        p1 = self.a[0] + self.a[1] * z  # 线性分子
        p2 = self.b[0] + self.b[1] * z + 1e-6  # 线性分母，加小值避免除零
        
        return (p1 + self.q1) / (p2 + self.q2)
    
    def get_z(self, X):
        """获取隐藏层输出z，用于有理函数参数优化"""
        with torch.no_grad():
            h = self.era(self.fc1(X))
            z = torch.matmul(h, self.alpha) + self.beta
        return z
    
    def rational_params(self):
        """返回有理函数参数供SciPy优化"""
        return torch.cat([self.a.data, self.b.data, self.q1.data, self.q2.data]).numpy()
    
    def update_rational_params(self, new_params):
        """更新有理函数参数"""
        with torch.no_grad():
            self.a.data = torch.from_numpy(new_params[:2]).float()
            self.b.data = torch.from_numpy(new_params[2:4]).float()
            self.q1.data = torch.from_numpy(new_params[4:5]).float()
            self.q2.data = torch.from_numpy(new_params[5:6]).float()


def alt_train_with_era(model, X, y_true, max_iter=100, verbose=True):
    """交替训练模型：先优化神经网络参数，再优化有理函数参数"""
    # 确保输入是torch张量
    if not isinstance(X, torch.Tensor):
        X = torch.tensor(X, dtype=torch.float32)
    if not isinstance(y_true, torch.Tensor):
        y_true = torch.tensor(y_true, dtype=torch.float32)
    
    # 阶段一优化器 - 只优化神经网络参数(不包括有理函数参数)
    optimizer = optim.Adam([
        {'params': model.fc1.parameters()},
        {'params': model.alpha},
        {'params': model.beta},
        {'params': model.era.parameters()}  # 包含ERA参数
    ], lr=1e-3)
    
    # 记录损失和结果
    losses = []
    results = {'training': {'loss_history': []}}
    
    for i in range(max_iter):
        # 阶段一：ADAM优化神经网络参数和ERA参数
        optimizer.zero_grad()
        y_pred = model(X)
        loss = torch.max(torch.abs(y_true - y_pred))  # 极小极大损失
        loss.backward()
        
        # 权重裁剪，稳定训练
        torch.nn.utils.clip_grad_value_(model.parameters(), clip_value=1.0)
        
        optimizer.step()
        
        # 阶段二：拟凸优化有理函数参数
        z = model.get_z(X).detach().numpy()
        
        # 定义优化目标函数：最小化最大误差
        def max_error(params):
            model.update_rational_params(params)
            with torch.no_grad():
                # 直接使用z计算，避免重复计算网络部分
                y_pred = (model.a[0] + model.a[1] * z + model.q1) / \
                         (model.b[0] + model.b[1] * z + model.q2 + 1e-6)
            return torch.max(torch.abs(torch.tensor(y_true.numpy() - y_pred.numpy()))).item()
        
        # 使用SLSQP优化器
        bounds = [(None, None)] * 6  # 所有参数无约束
        res = minimize(max_error, model.rational_params(), 
                      method='SLSQP', bounds=bounds, options={'maxiter': 50})
        
        # 更新有理函数参数
        model.update_rational_params(res.x)
        
        # 记录并打印损失
        current_loss = loss.item()
        losses.append(current_loss)
        results['training']['loss_history'].append(current_loss)
        
        if verbose and (i % 100 == 0):
            print(f"Iteration {i}, Loss: {current_loss:.6f}")
    
    return losses, results


if __name__ == "__main__":
    # 创建示例数据 
    X = torch.linspace(-3, 3, 100).reshape(-1, 1)  # 1D输入
    y_true = torch.sqrt(torch.abs(X - 0.25)).flatten() 
    
    # 创建模型
    model = RationalNNWithERA(input_dim=1, hidden_dim=10)
    
    # 训练模型
    print("开始训练模型...")
    losses, results = alt_train_with_era(model, X, y_true, max_iter=5000)
    
    # 测试模型
    with torch.no_grad():
        y_pred = model(X)
        final_error = torch.mean(torch.abs(y_pred - y_true)) 
        max_error = torch.max(torch.abs(y_pred - y_true))
        print(f"最终平均绝对误差: {final_error.item():.6f}")
        print(f"最终最大绝对误差: {max_error.item():.6f}")
    
    # 绘制训练损失对比图
    plt.figure(figsize=(10, 6))
    for name, metrics in results.items():
        plt.plot(metrics['loss_history'], label=name, alpha=0.7)
    plt.xlabel('Epoch')
    plt.ylabel('L1 Loss')
    plt.title('Training Loss Comparison')
    plt.legend()
    plt.grid(True)
    plt.show()
    
    # 添加预测结果与真实值对比图
    plt.figure(figsize=(10, 6))
    plt.scatter(X.numpy(), y_true.numpy(), label='真实值', alpha=0.5)
    plt.plot(X.numpy(), y_pred.numpy(), 'r-', label='预测值', linewidth=2)
    plt.xlabel('X')
    plt.ylabel('y')
    plt.title('预测结果与真实值对比')
    plt.legend()
    plt.grid(True)
    plt.show()
    

开始训练模型...
Iteration 0, Loss: 2.067779
Iteration 100, Loss: 0.785426
Iteration 200, Loss: 0.726846
Iteration 300, Loss: 0.673732
Iteration 400, Loss: 0.604152
Iteration 500, Loss: 0.497535
Iteration 600, Loss: 0.333528
Iteration 700, Loss: 0.174101
Iteration 800, Loss: 0.162232
Iteration 900, Loss: 0.155113
Iteration 1000, Loss: 0.150451
Iteration 1100, Loss: 0.141114
Iteration 1200, Loss: 0.141004
Iteration 1300, Loss: 0.131107
Iteration 1400, Loss: 0.127675
Iteration 1500, Loss: 0.121763
Iteration 1600, Loss: 0.121468
Iteration 1700, Loss: 0.116175
Iteration 1800, Loss: 0.110980
Iteration 1900, Loss: 0.106684
Iteration 2000, Loss: 0.107871
Iteration 2100, Loss: 0.108504
Iteration 2200, Loss: 0.096208
Iteration 2300, Loss: 0.092717
Iteration 2400, Loss: 0.090683
Iteration 2500, Loss: 0.090496
Iteration 2600, Loss: 0.087979
Iteration 2700, Loss: 0.090627
Iteration 2800, Loss: 0.090026
Iteration 2900, Loss: 0.097070
Iteration 3000, Loss: 0.100589
Iteration 3100, Loss: 0.100341
Iteration 

In [36]:
# 使用示例
if __name__ == "__main__":
    # 创建示例数据 (y = sin(x) + 噪声)
    X = torch.linspace(-3, 3, 100).reshape(-1, 1)  # 1D输入
    y_true = torch.sqrt(abs(X-0.25)).flatten() 
    
    # 创建模型
    model = RationalNNWithERA(input_dim=1, hidden_dim=10)
    
    # 训练模型
    print("开始训练模型...")
    losses = alt_train_with_era(model, X, y_true, max_iter=5000)
    
    # 测试模型
    with torch.no_grad():
        y_pred = model(X)
        final_error = torch.mean(torch.abs(y_pred - y_true)) 
        max_error = torch.max(torch.abs(y_pred - y_true))
        print(f"最终平均绝对误差: {final_error.item():.6f}")
        print(f"最终最大绝对误差: {max_error.item():.6f}")
        


开始训练模型...
Iteration 0, Loss: 2.964722
Iteration 100, Loss: 1.410508
Iteration 200, Loss: 0.751627
Iteration 300, Loss: 0.705901
Iteration 400, Loss: 0.653931
Iteration 500, Loss: 0.544633
Iteration 600, Loss: 0.321246
Iteration 700, Loss: 0.197323
Iteration 800, Loss: 0.123733
Iteration 900, Loss: 0.118224
Iteration 1000, Loss: 0.116037
Iteration 1100, Loss: 0.110836
Iteration 1200, Loss: 0.112091
Iteration 1300, Loss: 0.107680
Iteration 1400, Loss: 0.104002
Iteration 1500, Loss: 0.103416
Iteration 1600, Loss: 0.103736
Iteration 1700, Loss: 0.100750
Iteration 1800, Loss: 0.102083
Iteration 1900, Loss: 0.098886
Iteration 2000, Loss: 0.097263
Iteration 2100, Loss: 0.096305
Iteration 2200, Loss: 0.096663
Iteration 2300, Loss: 0.093228
Iteration 2400, Loss: 0.091961
Iteration 2500, Loss: 0.092279
Iteration 2600, Loss: 0.090198
Iteration 2700, Loss: 0.088538
Iteration 2800, Loss: 0.087040
Iteration 2900, Loss: 0.087295
Iteration 3000, Loss: 0.083484
Iteration 3100, Loss: 0.081216
Iteration 

In [5]:
import torch
import matplotlib.pyplot as plt

# 假设RationalNNWithERA和alt_train_with_era已定义
# class RationalNNWithERA(torch.nn.Module):
#     ...
# def alt_train_with_era(model, X, y_true, max_iter):
#     ...
#     return losses, results  # 假设该函数返回损失列表和结果字典

if __name__ == "__main__":
    # 创建示例数据 
    X = torch.linspace(-3, 3, 100).reshape(-1, 1)  # 1D输入
    y_true = torch.sqrt(torch.abs(X - 0.25)).flatten() 
    
    # 创建模型
    model = RationalNNWithERA(input_dim=1, hidden_dim=10)
    
    # 训练模型
    print("开始训练模型...")
    losses, results = alt_train_with_era(model, X, y_true, max_iter=500)
    
    # 测试模型
    with torch.no_grad():
        y_pred = model(X)
        final_error = torch.mean(torch.abs(y_pred - y_true)) 
        max_error = torch.max(torch.abs(y_pred - y_true))
        print(f"最终平均绝对误差: {final_error.item():.6f}")
        print(f"最终最大绝对误差: {max_error.item():.6f}")
    
    # 绘制训练损失对比图
    plt.figure(figsize=(10, 6))
    for name, metrics in results.items():
        plt.plot(metrics['loss_history'], label=name, alpha=0.7)
    plt.xlabel('Epoch')
    plt.ylabel('L1 Loss')
    plt.title('Training Loss Comparison')
    plt.legend()
    plt.grid(True)
    plt.show()
    
    # 添加预测结果与真实值对比图
    plt.figure(figsize=(10, 6))
    plt.scatter(X.numpy(), y_true.numpy(), label='真实值', alpha=0.5)
    plt.plot(X.numpy(), y_pred.numpy(), 'r-', label='预测值', linewidth=2)
    plt.xlabel('X')
    plt.ylabel('y')
    plt.title('预测结果与真实值对比')
    plt.legend()
    plt.grid(True)
    plt.show()

开始训练模型...
Iteration 0, Loss: 1.810448
Iteration 100, Loss: 1.055110
Iteration 200, Loss: 0.830615
Iteration 300, Loss: 0.718073
Iteration 400, Loss: 0.645430


ValueError: too many values to unpack (expected 2)

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# 保留原有的ERA激活函数和RationalNNWithERA网络结构
class ERA_PartialFractions(nn.Module):
    """基于部分分式分解的ERA激活函数"""
    def __init__(self, epsilon=1e-6):
        super(ERA_PartialFractions, self).__init__()
        self.epsilon = epsilon
        
        # 线性部分参数
        self.alpha = nn.Parameter(torch.randn(1))
        self.beta = nn.Parameter(torch.randn(1))
        
        # 部分分数参数
        self.p1 = nn.Parameter(torch.randn(1))
        self.q1 = nn.Parameter(torch.randn(1))
        self.p2 = nn.Parameter(torch.randn(1))
        self.q2 = nn.Parameter(torch.randn(1))
        
        # 分母参数 (确保初始时为正)
        self.c1 = nn.Parameter(torch.randn(1))
        self.d1 = nn.Parameter(torch.randn(1).abs() + 0.1)
        self.c2 = nn.Parameter(torch.randn(1))
        self.d2 = nn.Parameter(torch.randn(1).abs() + 0.1)
    
    def forward(self, x):
        # 分母多项式，确保不为零
        Q1 = (x - self.c1)**2 + self.d1**2 + self.epsilon
        Q2 = (x - self.c2)**2 + self.d2**2 + self.epsilon
        
        # 部分分数项
        frac1 = (self.p1 * x + self.q1) / Q1
        frac2 = (self.p2 * x + self.q2) / Q2
        
        # 完整ERA函数
        return self.alpha * x + self.beta + frac1 + frac2


class RationalNNWithERA(nn.Module):
    """结合ERA激活函数的有理神经网络"""
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        # 神经网络部分
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        
        # ERA激活函数
        self.era = ERA_PartialFractions()
        
        # 中间变量参数
        self.alpha = nn.Parameter(torch.randn(hidden_dim))
        self.beta = nn.Parameter(torch.randn(1))
        
        # 有理函数参数
        self.a = nn.Parameter(torch.randn(2))  # 线性分子
        self.b = nn.Parameter(torch.randn(2))  # 线性分母
        self.q1 = nn.Parameter(torch.randn(1))
        self.q2 = nn.Parameter(torch.ones(1))  # 初始为正，避免除零
        
        # 初始化策略：近似Swish函数
        self.initialize_as_swish()
    
    def initialize_as_swish(self):
        """将ERA初始化为近似Swish函数"""
        with torch.no_grad():
            # 近似Swish函数 x * sigmoid(x)
            self.era.alpha.fill_(0.5)   # 线性系数
            self.era.beta.fill_(0)      # 偏置
            
            # 部分分数参数设置
            self.era.p1.fill_(0.1)
            self.era.q1.fill_(0)
            self.era.p2.fill_(0)
            self.era.q2.fill_(0)
            
            # 分母参数设置为远离0的值，保证稳定性
            self.era.c1.fill_(0)
            self.era.d1.fill_(1)
            self.era.c2.fill_(0)
            self.era.d2.fill_(1)
            
            # 有理函数初始化为近似恒等映射
            self.a.data = torch.tensor([0.0, 1.0])
            self.b.data = torch.tensor([1.0, 0.0])
            self.q1.fill_(0.0)
            self.q2.fill_(1.0)
    
    def forward(self, x):
        # 前向传播：全连接 -> ERA激活 -> 中间变量 -> 有理函数输出
        h = self.era(self.fc1(x))
        z = torch.matmul(h, self.alpha) + self.beta  # 计算中间变量z
        
        # 有理函数部分
        p1 = self.a[0] + self.a[1] * z  # 线性分子
        p2 = self.b[0] + self.b[1] * z + 1e-6  # 线性分母，加小值避免除零
        
        return (p1 + self.q1) / (p2 + self.q2)
    
    def get_z(self, X):
        """获取隐藏层输出z，用于有理函数参数优化"""
        with torch.no_grad():
            h = self.era(self.fc1(X))
            z = torch.matmul(h, self.alpha) + self.beta
        return z
    
    def rational_params(self):
        """返回有理函数参数供SciPy优化"""
        return torch.cat([self.a.data, self.b.data, self.q1.data, self.q2.data]).numpy()
    
    def update_rational_params(self, new_params):
        """更新有理函数参数"""
        with torch.no_grad():
            self.a.data = torch.from_numpy(new_params[:2]).float()
            self.b.data = torch.from_numpy(new_params[2:4]).float()
            self.q1.data = torch.from_numpy(new_params[4:5]).float()
            self.q2.data = torch.from_numpy(new_params[5:6]).float()


# 扩展网络用于求解H-J方程
class HJ_PINN(nn.Module):
    """求解Hamilton-Jacobi方程的PINN"""
    def __init__(self, input_dim=2, hidden_dim=20):
        super(HJ_PINN, self).__init__()
        # 使用RationalNNWithERA作为基础网络
        self.rational_nn = RationalNNWithERA(input_dim, hidden_dim)
        
    def forward(self, x):
        """输入应为(x, t)张量，输出为u(x, t)"""
        return self.rational_nn(x)
    
    def compute_gradients(self, x, create_graph=True):
        """计算u及其梯度"""
        x.requires_grad_(True)
        u = self.forward(x)
        
        # 计算梯度: du/dx, du/dt
        grad_u = torch.autograd.grad(
            u, x, 
            grad_outputs=torch.ones_like(u),
            create_graph=create_graph,
            retain_graph=True
        )[0]
        
        # 分离空间和时间梯度
        u_x = grad_u[:, 0:1]
        u_t = grad_u[:, 1:2]
        
        return u, u_x, u_t


def hj_equation_loss(model, x, hamiltonian, w_physics=1.0, w_init=1.0, w_bc=1.0):
    """计算H-J方程的损失函数"""
    # 1. 物理方程残差
    _, u_x, u_t = model.compute_gradients(x)
    residual = u_t + hamiltonian(u_x)
    physics_loss = torch.mean(residual**2)
    
    # 2. 初始条件损失 (t=0)
    x_init = x[x[:, 1] == 0]  # 选择t=0的点
    if len(x_init) > 0:
        u_init_pred = model(x_init)
        # 假设初始条件为 u(x,0) = |x|
        u_init_true = torch.abs(x_init[:, 0:1])
        init_loss = torch.mean((u_init_pred - u_init_true)**2)
    else:
        init_loss = torch.tensor(0.0)
    
    # 3. 边界条件损失 (x=±1)
    x_bc = x[(x[:, 0] == -1) | (x[:, 0] == 1)]
    if len(x_bc) > 0:
        u_bc_pred = model(x_bc)
        # 假设边界条件为 u(±1,t) = 1
        u_bc_true = torch.ones_like(u_bc_pred)
        bc_loss = torch.mean((u_bc_pred - u_bc_true)**2)
    else:
        bc_loss = torch.tensor(0.0)
    
    # 总损失
    total_loss = w_physics * physics_loss + w_init * init_loss + w_bc * bc_loss
    return total_loss, physics_loss, init_loss, bc_loss


def train_hj_pinn(model, domain, hamiltonian, epochs=1000, lr=0.001):
    """训练H-J方程的PINN"""
    # 创建训练数据
    n_points = 1000
    x = torch.linspace(domain['x_min'], domain['x_max'], n_points)
    t = torch.linspace(domain['t_min'], domain['t_max'], n_points)
    X, T = torch.meshgrid(x, t, indexing='ij')
    x_train = torch.stack([X.ravel(), T.ravel()], dim=1).float()
    
    # 优化器
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # 训练循环
    losses = []
    physics_losses = []
    init_losses = []
    bc_losses = []
    
    for epoch in range(epochs):
        optimizer.zero_grad()
        
        # 计算损失
        total_loss, physics_loss, init_loss, bc_loss = hj_equation_loss(
            model, x_train, hamiltonian
        )
        
        # 反向传播
        total_loss.backward()
        optimizer.step()
        
        # 记录损失
        losses.append(total_loss.item())
        physics_losses.append(physics_loss.item())
        init_losses.append(init_loss.item() if torch.is_tensor(init_loss) else init_loss)
        bc_losses.append(bc_loss.item() if torch.is_tensor(bc_loss) else bc_loss)
        
        if epoch % 100 == 0:
            print(f"Epoch {epoch}/{epochs}, Total Loss: {total_loss.item():.6f}, "
                  f"Physics Loss: {physics_loss.item():.6f}, "
                  f"Init Loss: {init_loss.item() if torch.is_tensor(init_loss) else init_loss:.6f}, "
                  f"BC Loss: {bc_loss.item() if torch.is_tensor(bc_loss) else bc_loss:.6f}")
    
    # 绘制损失曲线
    plt.figure(figsize=(10, 6))
    plt.plot(losses, label='Total Loss')
    plt.plot(physics_losses, label='Physics Loss')
    plt.plot(init_losses, label='Initial Condition Loss')
    plt.plot(bc_losses, label='Boundary Condition Loss')
    plt.yscale('log')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training Losses')
    plt.legend()
    plt.grid(True)
    plt.show()
    
    return model


def visualize_solution(model, domain):
    """可视化求解结果"""
    # 创建网格
    n_points = 100
    x = torch.linspace(domain['x_min'], domain['x_max'], n_points)
    t = torch.linspace(domain['t_min'], domain['t_max'], n_points)
    X, T = torch.meshgrid(x, t, indexing='ij')
    grid = torch.stack([X.ravel(), T.ravel()], dim=1).float()
    
    # 预测
    with torch.no_grad():
        u_pred = model(grid)
    U = u_pred.reshape(n_points, n_points).numpy()
    
    # 3D图
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(X.numpy(), T.numpy(), U, cmap='viridis', rstride=1, cstride=1, alpha=0.8)
    
    # 添加激波位置 (示例)
    shock_x = np.zeros_like(T.numpy())
    shock_t = T.numpy()
    shock_u = np.abs(shock_x)
    ax.plot(shock_x, shock_t, shock_u, 'r-', linewidth=2, label='Shock')
    
    ax.set_xlabel('x')
    ax.set_ylabel('t')
    ax.set_zlabel('u(x,t)')
    ax.set_title('Solution of Hamilton-Jacobi Equation with Shock')
    fig.colorbar(surf, ax=ax, shrink=0.5, aspect=5)
    plt.legend()
    plt.show()
    
    # 2D切片图
    plt.figure(figsize=(12, 6))
    
    # 不同时刻的切片
    t_slices = [0.0, 0.25, 0.5]
    colors = ['b', 'g', 'r']
    
    for i, t_val in enumerate(t_slices):
        # 找到最接近的时间索引
        t_idx = np.argmin(np.abs(t.numpy() - t_val))
        plt.plot(x.numpy(), U[:, t_idx], colors[i], label=f't={t_val:.2f}')
    
    plt.xlabel('x')
    plt.ylabel('u(x,t)')
    plt.title('Solution Slices at Different Times')
    plt.legend()
    plt.grid(True)
    plt.show()


# 主程序
if __name__ == "__main__":
    # 定义求解区域
    domain = {
        'x_min': -1.0,
        'x_max': 1.0,
        't_min': 0.0,
        't_max': 0.5
    }
    
    # 定义Hamiltonian函数 (示例：H(p) = |p|)
    def hamiltonian(p):
        return torch.abs(p)
    
    # 初始化模型
    model = HJ_PINN(input_dim=2, hidden_dim=20)
    
    # 训练模型
    trained_model = train_hj_pinn(
        model, 
        domain, 
        hamiltonian,
        epochs=3000,
        lr=0.001
    )
    
    # 可视化结果
    visualize_solution(trained_model, domain)
    
    # 保存模型
    torch.save(trained_model.state_dict(), 'hj_pinn_model.pth')

Epoch 0/3000, Total Loss: 0.638679, Physics Loss: 0.071761, Init Loss: 0.469921, BC Loss: 0.096997


KeyboardInterrupt: 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from tqdm import tqdm

# 基线模型1: 标准全连接神经网络 (ReLU激活)
class FCNet_ReLU(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)  # 回归任务输出1维
        )
        
    def forward(self, x):
        return self.model(x).flatten()  # 展平输出


# 基线模型2: 全连接网络 (Swish激活)
class FCNet_Swish(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.hidden1 = nn.Linear(input_dim, hidden_dim)
        self.hidden2 = nn.Linear(hidden_dim, hidden_dim)
        self.output = nn.Linear(hidden_dim, 1)
        
    def forward(self, x):
        x = self.hidden1(x)
        x = x * torch.sigmoid(x)  # Swish激活
        x = self.hidden2(x)
        x = x * torch.sigmoid(x)  # Swish激活
        return self.output(x).flatten()


# 基线模型3: 简化版有理网络 (仅使用基础激活函数)
class SimpleRationalNet(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.alpha = nn.Parameter(torch.randn(hidden_dim))
        self.beta = nn.Parameter(torch.randn(1))
        
        # 简单有理函数输出层
        self.a = nn.Parameter(torch.tensor([0.0, 1.0]))  # 分子系数
        self.b = nn.Parameter(torch.tensor([1.0, 0.0]))  # 分母系数
        
    def forward(self, x):
        h = self.relu(self.fc1(x))
        z = torch.matmul(h, self.alpha) + self.beta
        return (self.a[0] + self.a[1] * z) / (self.b[0] + self.b[1] * z + 1e-6)


# 统一训练函数 (适用于所有基线模型)
def train_baseline(model, X, y_true, epochs=5000, lr=1e-3, verbose=True):
    if not isinstance(X, torch.Tensor):
        X = torch.tensor(X, dtype=torch.float32)
    if not isinstance(y_true, torch.Tensor):
        y_true = torch.tensor(y_true, dtype=torch.float32)
    
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.L1Loss()  # 使用L1损失与原模型的MAE对比
    losses = []
    
    for epoch in tqdm(range(epochs), disable=not verbose):
        optimizer.zero_grad()
        y_pred = model(X)
        loss = criterion(y_pred, y_true)
        loss.backward()
        optimizer.step()
        
        losses.append(loss.item())
        if verbose and (epoch % 50 == 0):
            print(f"Epoch {epoch}, Loss: {loss.item():.6f}")
    
    return losses


# 模型比较实验
if __name__ == "__main__":
    # 生成与之前相同的数据集
    X = torch.linspace(-3, 3, 100).reshape(-1, 1)
    y_true = torch.sqrt(abs(X-0.25)).flatten()  + 0.1 * torch.randn(100)
    
    # 初始化所有模型 (保持相同的隐藏层维度)
    hidden_dim = 10
    models = {
        "FC-ReLU": FCNet_ReLU(1, hidden_dim),
        "FC-Swish": FCNet_Swish(1, hidden_dim),
        "Simple-Rational": SimpleRationalNet(1, hidden_dim)
    }
    
    # 训练并评估所有模型
    results = {}
    for name, model in models.items():
        print(f"\n训练 {name}...")
        losses = train_baseline(model, X, y_true, epochs=5000)
        
        # 评估最终性能
        with torch.no_grad():
            y_pred = model(X)
            mae = torch.mean(torch.abs(y_pred - y_true)).item()
            max_error = torch.max(torch.abs(y_pred - y_true)).item()
        
        results[name] = {
            "final_mae": mae,
            "max_error": max_error,
            "last_loss": losses[-1]
        }
        print(f"{name} 结果: MAE={mae:.6f}, 最大误差={max_error:.6f}")
    
    # 打印比较结果
    print("\n===== 模型比较结果 =====")
    for name, metrics in results.items():
        print(f"{name}:")
        print(f"  平均绝对误差: {metrics['final_mae']:.6f}")
        print(f"  最大误差: {metrics['max_error']:.6f}")
    

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

# 保留原有的ERA激活函数和RationalNNWithERA网络结构
class ERA_PartialFractions(nn.Module):
    """基于部分分式分解的ERA激活函数"""
    def __init__(self, epsilon=1e-6):
        super(ERA_PartialFractions, self).__init__()
        self.epsilon = epsilon
        
        # 线性部分参数
        self.alpha = nn.Parameter(torch.randn(1))
        self.beta = nn.Parameter(torch.randn(1))
        
        # 部分分数参数
        self.p1 = nn.Parameter(torch.randn(1))
        self.q1 = nn.Parameter(torch.randn(1))
        self.p2 = nn.Parameter(torch.randn(1))
        self.q2 = nn.Parameter(torch.randn(1))
        
        # 分母参数 (确保初始时为正)
        self.c1 = nn.Parameter(torch.randn(1))
        self.d1 = nn.Parameter(torch.randn(1).abs() + 0.1)
        self.c2 = nn.Parameter(torch.randn(1))
        self.d2 = nn.Parameter(torch.randn(1).abs() + 0.1)
    
    def forward(self, x):
        # 分母多项式，确保不为零
        Q1 = (x - self.c1)**2 + self.d1**2 + self.epsilon
        Q2 = (x - self.c2)**2 + self.d2**2 + self.epsilon
        
        # 部分分数项
        frac1 = (self.p1 * x + self.q1) / Q1
        frac2 = (self.p2 * x + self.q2) / Q2
        
        # 完整ERA函数
        return self.alpha * x + self.beta + frac1 + frac2


class RationalNNWithERA(nn.Module):
    """结合ERA激活函数的有理神经网络"""
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        # 神经网络部分
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        
        # ERA激活函数
        self.era = ERA_PartialFractions()
        
        # 中间变量参数
        self.alpha = nn.Parameter(torch.randn(hidden_dim))
        self.beta = nn.Parameter(torch.randn(1))
        
        # 有理函数参数
        self.a = nn.Parameter(torch.randn(2))  # 线性分子
        self.b = nn.Parameter(torch.randn(2))  # 线性分母
        self.q1 = nn.Parameter(torch.randn(1))
        self.q2 = nn.Parameter(torch.ones(1))  # 初始为正，避免除零
        
        # 初始化策略：近似Swish函数
        self.initialize_as_swish()
    
    def initialize_as_swish(self):
        """将ERA初始化为近似Swish函数"""
        with torch.no_grad():
            # 近似Swish函数 x * sigmoid(x)
            self.era.alpha.fill_(0.5)   # 线性系数
            self.era.beta.fill_(0)      # 偏置
            
            # 部分分数参数设置
            self.era.p1.fill_(0.1)
            self.era.q1.fill_(0)
            self.era.p2.fill_(0)
            self.era.q2.fill_(0)
            
            # 分母参数设置为远离0的值，保证稳定性
            self.era.c1.fill_(0)
            self.era.d1.fill_(1)
            self.era.c2.fill_(0)
            self.era.d2.fill_(1)
            
            # 有理函数初始化为近似恒等映射
            self.a.data = torch.tensor([0.0, 1.0])
            self.b.data = torch.tensor([1.0, 0.0])
            self.q1.fill_(0.0)
            self.q2.fill_(1.0)
    
    def forward(self, x):
        # 前向传播：全连接 -> ERA激活 -> 中间变量 -> 有理函数输出
        h = self.era(self.fc1(x))
        z = torch.matmul(h, self.alpha) + self.beta  # 计算中间变量z
        
        # 有理函数部分
        p1 = self.a[0] + self.a[1] * z  # 线性分子
        p2 = self.b[0] + self.b[1] * z + 1e-6  # 线性分母，加小值避免除零
        
        return (p1 + self.q1) / (p2 + self.q2)
    
    def get_z(self, X):
        """获取隐藏层输出z，用于有理函数参数优化"""
        with torch.no_grad():
            h = self.era(self.fc1(X))
            z = torch.matmul(h, self.alpha) + self.beta
        return z
    
    def rational_params(self):
        """返回有理函数参数供SciPy优化"""
        return torch.cat([self.a.data, self.b.data, self.q1.data, self.q2.data]).numpy()
    
    def update_rational_params(self, new_params):
        """更新有理函数参数"""
        with torch.no_grad():
            self.a.data = torch.from_numpy(new_params[:2]).float()
            self.b.data = torch.from_numpy(new_params[2:4]).float()
            self.q1.data = torch.from_numpy(new_params[4:5]).float()
            self.q2.data = torch.from_numpy(new_params[5:6]).float()


# 扩展网络用于求解H-J方程
class HJ_PINN(nn.Module):
    """求解Hamilton-Jacobi方程的PINN"""
    def __init__(self, input_dim=2, hidden_dim=20):
        super(HJ_PINN, self).__init__()
        # 使用RationalNNWithERA作为基础网络
        self.rational_nn = RationalNNWithERA(input_dim, hidden_dim)
        
    def forward(self, x):
        """输入应为(x, t)张量，输出为u(x, t)"""
        return self.rational_nn(x)
    
    def compute_gradients(self, x, create_graph=True):
        """计算u及其梯度"""
        x.requires_grad_(True)
        u = self.forward(x)
        
        # 计算梯度: du/dx, du/dt
        grad_u = torch.autograd.grad(
            u, x, 
            grad_outputs=torch.ones_like(u),
            create_graph=create_graph,
            retain_graph=True
        )[0]
        
        # 分离空间和时间梯度
        u_x = grad_u[:, 0:1]
        u_t = grad_u[:, 1:2]
        
        return u, u_x, u_t


def hj_equation_loss(model, x, hamiltonian, w_physics=1.0, w_init=1.0, w_bc=1.0):
    """计算H-J方程的损失函数"""
    # 1. 物理方程残差
    _, u_x, u_t = model.compute_gradients(x)
    residual = u_t + hamiltonian(u_x)
    physics_loss = torch.mean(residual**2)
    
    # 2. 初始条件损失 (t=0)
    x_init = x[x[:, 1] == 0]  # 选择t=0的点
    if len(x_init) > 0:
        u_init_pred = model(x_init)
        # 假设初始条件为 u(x,0) = |x|
        u_init_true = torch.abs(x_init[:, 0:1])
        init_loss = torch.mean((u_init_pred - u_init_true)**2)
    else:
        init_loss = torch.tensor(0.0)
    
    # 3. 边界条件损失 (x=±1)
    x_bc = x[(x[:, 0] == -1) | (x[:, 0] == 1)]
    if len(x_bc) > 0:
        u_bc_pred = model(x_bc)
        # 假设边界条件为 u(±1,t) = 1
        u_bc_true = torch.ones_like(u_bc_pred)
        bc_loss = torch.mean((u_bc_pred - u_bc_true)**2)
    else:
        bc_loss = torch.tensor(0.0)
    
    # 总损失
    total_loss = w_physics * physics_loss + w_init * init_loss + w_bc * bc_loss
    return total_loss, physics_loss, init_loss, bc_loss


def train_hj_pinn(model, domain, hamiltonian, epochs=100, lr=0.001):
    """训练H-J方程的PINN"""
    # 创建训练数据
    n_points = 100
    x = torch.linspace(domain['x_min'], domain['x_max'], n_points)
    t = torch.linspace(domain['t_min'], domain['t_max'], n_points)
    X, T = torch.meshgrid(x, t, indexing='ij')
    x_train = torch.stack([X.ravel(), T.ravel()], dim=1).float()
    
    # 优化器
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # 训练循环
    losses = []
    physics_losses = []
    init_losses = []
    bc_losses = []
    
    for epoch in range(epochs):
        optimizer.zero_grad()
        
        # 计算损失
        total_loss, physics_loss, init_loss, bc_loss = hj_equation_loss(
            model, x_train, hamiltonian
        )
        
        # 反向传播
        total_loss.backward()
        optimizer.step()
        
        # 记录损失
        losses.append(total_loss.item())
        physics_losses.append(physics_loss.item())
        init_losses.append(init_loss.item() if torch.is_tensor(init_loss) else init_loss)
        bc_losses.append(bc_loss.item() if torch.is_tensor(bc_loss) else bc_loss)
        
        if epoch % 10 == 0:
            print(f"Epoch {epoch}/{epochs}, Total Loss: {total_loss.item():.6f}, "
                  f"Physics Loss: {physics_loss.item():.6f}, "
                  f"Init Loss: {init_loss.item() if torch.is_tensor(init_loss) else init_loss:.6f}, "
                  f"BC Loss: {bc_loss.item() if torch.is_tensor(bc_loss) else bc_loss:.6f}")
    
    # 绘制损失曲线
    plt.figure(figsize=(10, 6))
    plt.plot(losses, label='Total Loss')
    plt.plot(physics_losses, label='Physics Loss')
    plt.plot(init_losses, label='Initial Condition Loss')
    plt.plot(bc_losses, label='Boundary Condition Loss')
    plt.yscale('log')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training Losses')
    plt.legend()
    plt.grid(True)
    plt.show()
    
    return model


def visualize_solution(model, domain):
    """可视化求解结果"""
    # 创建网格
    n_points = 100
    x = torch.linspace(domain['x_min'], domain['x_max'], n_points)
    t = torch.linspace(domain['t_min'], domain['t_max'], n_points)
    X, T = torch.meshgrid(x, t, indexing='ij')
    grid = torch.stack([X.ravel(), T.ravel()], dim=1).float()
    
    # 预测
    with torch.no_grad():
        u_pred = model(grid)
    U = u_pred.reshape(n_points, n_points).numpy()
    
    # 3D图
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(X.numpy(), T.numpy(), U, cmap='viridis', rstride=1, cstride=1, alpha=0.8)
    
    # 添加激波位置 (示例)
    shock_x = np.zeros_like(T.numpy())
    shock_t = T.numpy()
    shock_u = np.abs(shock_x)
    ax.plot(shock_x, shock_t, shock_u, 'r-', linewidth=2, label='Shock')
    
    ax.set_xlabel('x')
    ax.set_ylabel('t')
    ax.set_zlabel('u(x,t)')
    ax.set_title('Solution of Hamilton-Jacobi Equation with Shock')
    fig.colorbar(surf, ax=ax, shrink=0.5, aspect=5)
    plt.legend()
    plt.show()
    
    # 2D切片图
    plt.figure(figsize=(12, 6))
    
    # 不同时刻的切片
    t_slices = [0.0, 0.25, 0.5]
    colors = ['b', 'g', 'r']
    
    for i, t_val in enumerate(t_slices):
        # 找到最接近的时间索引
        t_idx = np.argmin(np.abs(t.numpy() - t_val))
        plt.plot(x.numpy(), U[:, t_idx], colors[i], label=f't={t_val:.2f}')
    
    plt.xlabel('x')
    plt.ylabel('u(x,t)')
    plt.title('Solution Slices at Different Times')
    plt.legend()
    plt.grid(True)
    plt.show()


# 主程序
if __name__ == "__main__":
    # 定义求解区域
    domain = {
        'x_min': -1.0,
        'x_max': 1.0,
        't_min': 0.0,
        't_max': 0.5
    }
    
    # 定义Hamiltonian函数 (示例：H(p) = |p|)
    def hamiltonian(p):
        return torch.abs(p)
    
    # 初始化模型
    model = HJ_PINN(input_dim=2, hidden_dim=20)
    
    # 训练模型
    trained_model = train_hj_pinn(
        model, 
        domain, 
        hamiltonian,
        epochs=20,
        lr=0.001
    )
    
    # 可视化结果
    visualize_solution(trained_model, domain)
    
    # 保存模型
    torch.save(trained_model.state_dict(), 'hj_pinn_model.pth')

Epoch 0/20, Total Loss: 0.881831, Physics Loss: 0.127350, Init Loss: 0.653668, BC Loss: 0.100812
Epoch 10/20, Total Loss: 0.365658, Physics Loss: 0.066901, Init Loss: 0.297340, BC Loss: 0.001418


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import gc
gc.collect()  # 手动触发垃圾回收
torch.cuda.empty_cache()  # 如果使用GPU，清理缓存
# 使用ReLU激活函数的简化网络
class SimpleNN(nn.Module):
    """简化的神经网络，使用ReLU激活函数"""
    def __init__(self, input_dim=2, hidden_dim=16):
        super(SimpleNN, self).__init__()
        # 简化网络结构，减少参数数量
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)  # 直接输出u(x,t)
        )
        
        # 简单初始化
        self._initialize_weights()
    
    def _initialize_weights(self):
        """初始化权重以加速收敛"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.constant_(m.bias, 0.1)
    
    def forward(self, x):
        return self.layers(x)


class HJ_PINN(nn.Module):
    """求解Hamilton-Jacobi方程的PINN，使用ReLU激活函数"""
    def __init__(self, input_dim=2, hidden_dim=16):
        super(HJ_PINN, self).__init__()
        self.nn = SimpleNN(input_dim, hidden_dim)
        
    def forward(self, x):
        """输入应为(x, t)张量，输出为u(x, t)"""
        return self.nn(x)
    
    def compute_gradients(self, x, create_graph=True):
        """计算u及其梯度"""
        x.requires_grad_(True)
        u = self.forward(x)
        
        # 计算梯度: du/dx, du/dt
        grad_u = torch.autograd.grad(
            u, x, 
            grad_outputs=torch.ones_like(u),
            create_graph=create_graph,
            retain_graph=True
        )[0]
        
        # 分离空间和时间梯度
        u_x = grad_u[:, 0:1]  # 对x的偏导
        u_t = grad_u[:, 1:2]  # 对t的偏导
        
        return u, u_x, u_t


def hj_equation_loss(model, x, t, hamiltonian, w_physics=1.0, w_init=1.0, w_bc=1.0):
    """计算H-J方程的损失函数，优化边界条件检测"""
    # 合并x和t为输入张量
    x_train = torch.cat([x, t], dim=1)
    
    # 1. 物理方程残差
    _, u_x, u_t = model.compute_gradients(x_train)
    residual = u_t + hamiltonian(u_x)
    physics_loss = torch.mean(residual**2)
    
    # 2. 初始条件损失 (t≈0) - 使用近似比较而非精确等于
    t_zero_mask = (t < 1e-5).squeeze()  # 找到接近t=0的点
    x_init = x_train[t_zero_mask]
    if len(x_init) > 0:
        u_init_pred = model(x_init)
        # 初始条件为 u(x,0) = |x|
        u_init_true = torch.abs(x_init[:, 0:1])  # x坐标在第一列
        init_loss = torch.mean((u_init_pred - u_init_true)**2)
    else:
        init_loss = torch.tensor(0.0)
    
    # 3. 边界条件损失 (x≈±1) - 使用近似比较
    x_min_mask = (torch.abs(x + 1.0) < 1e-5).squeeze()  # x≈-1
    x_max_mask = (torch.abs(x - 1.0) < 1e-5).squeeze()  # x≈1
    bc_mask = x_min_mask | x_max_mask
    x_bc = x_train[bc_mask]
    
    if len(x_bc) > 0:
        u_bc_pred = model(x_bc)
        # 边界条件为 u(±1,t) = 1
        u_bc_true = torch.ones_like(u_bc_pred)
        bc_loss = torch.mean((u_bc_pred - u_bc_true)** 2)
    else:
        bc_loss = torch.tensor(0.0)
    
    # 总损失
    total_loss = w_physics * physics_loss + w_init * init_loss + w_bc * bc_loss
    return total_loss, physics_loss, init_loss, bc_loss


def train_hj_pinn(model, domain, hamiltonian, epochs=100, lr=0.005):
    """训练H-J方程的PINN，优化以加速训练"""
    # 减少训练点数以加快训练 (从1000x1000减少到100x100)
    n_points = 100
    x = torch.linspace(domain['x_min'], domain['x_max'], n_points).unsqueeze(1)
    t = torch.linspace(domain['t_min'], domain['t_max'], n_points).unsqueeze(1)
    
    # 创建网格数据
    X = x.repeat(1, n_points).view(-1, 1)
    T = t.repeat(n_points, 1)
    
    # 优化器使用Adam，学习率稍大以加速收敛
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # 学习率调度器，动态调整学习率
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)
    
    # 训练循环
    losses = []
    physics_losses = []
    init_losses = []
    bc_losses = []
    
    for epoch in range(epochs):
        optimizer.zero_grad()
        
        # 计算损失
        total_loss, physics_loss, init_loss, bc_loss = hj_equation_loss(
            model, X, T, hamiltonian
        )
        
        # 反向传播
        total_loss.backward()
        optimizer.step()
        scheduler.step()
        
        # 记录损失
        losses.append(total_loss.item())
        physics_losses.append(physics_loss.item())
        init_losses.append(init_loss.item() if torch.is_tensor(init_loss) else init_loss)
        bc_losses.append(bc_loss.item() if torch.is_tensor(bc_loss) else bc_loss)
        
        # 减少打印频率以加快训练
        if epoch % 10 == 0:
            print(f"Epoch {epoch}/{epochs}, Total Loss: {total_loss.item():.6f}, "
                  f"Physics Loss: {physics_loss.item():.6f}")
    
    # 绘制损失曲线
    plt.figure(figsize=(10, 6))
    plt.plot(losses, label='Total Loss')
    plt.plot(physics_losses, label='Physics Loss')
    plt.plot(init_losses, label='Initial Condition Loss')
    plt.plot(bc_losses, label='Boundary Condition Loss')
    plt.yscale('log')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training Losses')
    plt.legend()
    plt.grid(True)
    plt.show()
    
    return model


def visualize_solution(model, domain):
    """可视化求解结果"""
    # 创建网格
    n_points = 100
    x = torch.linspace(domain['x_min'], domain['x_max'], n_points)
    t = torch.linspace(domain['t_min'], domain['t_max'], n_points)
    X, T = torch.meshgrid(x, t, indexing='ij')
    grid = torch.stack([X.ravel(), T.ravel()], dim=1).float()
    
    # 预测
    with torch.no_grad():  # 禁用梯度计算以加快预测
        u_pred = model(grid)
    U = u_pred.reshape(n_points, n_points).numpy()
    
    # 3D图
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(X.numpy(), T.numpy(), U, cmap='viridis', rstride=1, cstride=1, alpha=0.8)
    
    # 添加激波位置
    shock_x = np.zeros_like(T.numpy())
    shock_t = T.numpy()
    shock_u = np.abs(shock_x)
    ax.plot(shock_x, shock_t, shock_u, 'r-', linewidth=2, label='Shock')
    
    ax.set_xlabel('x')
    ax.set_ylabel('t')
    ax.set_zlabel('u(x,t)')
    ax.set_title('Solution of Hamilton-Jacobi Equation with Shock')
    fig.colorbar(surf, ax=ax, shrink=0.5, aspect=5)
    plt.legend()
    plt.show()
    
    # 2D切片图
    plt.figure(figsize=(12, 6))
    
    # 不同时刻的切片
    t_slices = [0.0, 0.25, 0.5]
    colors = ['b', 'g', 'r']
    
    for i, t_val in enumerate(t_slices):
        # 找到最接近的时间索引
        t_idx = np.argmin(np.abs(t.numpy() - t_val))
        plt.plot(x.numpy(), U[:, t_idx], colors[i], label=f't={t_val:.2f}')
    
    plt.xlabel('x')
    plt.ylabel('u(x,t)')
    plt.title('Solution Slices at Different Times')
    plt.legend()
    plt.grid(True)
    plt.show()


# 主程序
if __name__ == "__main__":
    # 定义求解区域
    domain = {
        'x_min': -1.0,
        'x_max': 1.0,
        't_min': 0.0,
        't_max': 0.5
    }
    
    # 定义Hamiltonian函数 (H(p) = |p|)
    def hamiltonian(p):
        return torch.abs(p)
    
    # 初始化模型 - 使用更小的隐藏层
    model = HJ_PINN(input_dim=2, hidden_dim=16)
    
    # 训练模型 - 减少迭代次数
    trained_model = train_hj_pinn(
        model, 
        domain, 
        hamiltonian,
        epochs=1000,  # 大幅减少迭代次数
        lr=0.005
    )
    
    # 可视化结果
    visualize_solution(trained_model, domain)
    
    # 保存模型
    torch.save(trained_model.state_dict(), 'hj_pinn_relu_model.pth')


Epoch 0/1000, Total Loss: 2.327219, Physics Loss: 0.064342
Epoch 10/1000, Total Loss: 0.315389, Physics Loss: 0.013928
Epoch 20/1000, Total Loss: 0.241118, Physics Loss: 0.074717
Epoch 30/1000, Total Loss: 0.209755, Physics Loss: 0.069513
Epoch 40/1000, Total Loss: 0.196230, Physics Loss: 0.050912
Epoch 50/1000, Total Loss: 0.159547, Physics Loss: 0.025687
Epoch 60/1000, Total Loss: 0.145386, Physics Loss: 0.031581
Epoch 70/1000, Total Loss: 0.138562, Physics Loss: 0.033992
Epoch 80/1000, Total Loss: 0.130446, Physics Loss: 0.025385
Epoch 90/1000, Total Loss: 0.124995, Physics Loss: 0.023900
Epoch 100/1000, Total Loss: 0.119819, Physics Loss: 0.024150
Epoch 110/1000, Total Loss: 0.117294, Physics Loss: 0.024520
Epoch 120/1000, Total Loss: 0.114714, Physics Loss: 0.024828
Epoch 130/1000, Total Loss: 0.111522, Physics Loss: 0.024863
Epoch 140/1000, Total Loss: 0.108327, Physics Loss: 0.024672
Epoch 150/1000, Total Loss: 0.105243, Physics Loss: 0.024530
Epoch 160/1000, Total Loss: 0.10376

In [3]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
def check_gpu_and_cuda():
    """检查GPU是否可用并显示CUDA相关信息"""
    print("=== 系统信息 ===")
    print(f"Python版本: {sys.version.split()[0]}")
    print(f"PyTorch版本: {torch.__version__}")
    
    print("\n=== GPU检测 ===")
    # 检查是否有可用的GPU
    if torch.cuda.is_available():
        # 获取GPU数量
        gpu_count = torch.cuda.device_count()
        print(f"发现 {gpu_count} 个可用的GPU")
        
        # 显示每个GPU的信息
        for i in range(gpu_count):
            print(f"\nGPU {i}:")
            print(f"  名称: {torch.cuda.get_device_name(i)}")
            print(f"  计算能力: {torch.cuda.get_device_capability(i)}")
            print(f"  内存总量: {torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GB")
        
        # 显示当前CUDA版本
        print(f"\nCUDA版本: {torch.version.cuda}")
        
        # 设置默认设备为第一个GPU
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        print(f"\n默认设备已设置为: {device}")
        return device
    else:
        print("未检测到可用的GPU或CUDA环境")
        print("将使用CPU进行计算")
        return torch.device("cpu")